In [3]:
import numpy as np
import plotly.figure_factory as ff
import plotly.graph_objects as go

In [23]:
def compute_multispring_fields(X_grid, Y_grid, spring_list):
    """
    Computes total potential energy and net force field for N springs.
    Delta points from mass (X, Y) to each anchor.
    """
    U_total = np.zeros_like(X_grid, dtype=float)
    Fx_total = np.zeros_like(X_grid, dtype=float)
    Fy_total = np.zeros_like(X_grid, dtype=float)

    for s in spring_list:
        ax, ay = s["anchor"]
        k = s["k"]
        L0 = s["L0"]

        # Vector differences pointing from mass to anchor
        dx = ax - X_grid
        dy = ay - Y_grid
        L = np.sqrt(dx**2 + dy**2) + 1e-9  # Avoid division by zero

        # Potential: 0.5 * k * (L - L0)^2
        U_total += 0.5 * k * (L - L0) ** 2

        # Force: F = k * (1 - L0 / L) * Delta
        Fx_total += k * (1.0 - L0 / L) * dx
        Fy_total += k * (1.0 - L0 / L) * dy

    return U_total, Fx_total, Fy_total

In [134]:
def plot_potential(
    springs,
    point_masses,
    scale=1.0,
    arrow_scale=1.0,
    force_arrow_head=2,
    force_arrow_size=1.5,
):
    # Ensure point_masses is a 2D numpy array: shape (N_masses, 2)
    masses = np.atleast_2d(point_masses)

    # ---------------------------------------------------------
    # Dynamic Domain & Grid Setup
    # ---------------------------------------------------------
    all_anchors = np.array([s["anchor"] for s in springs])
    all_points = np.vstack([all_anchors, masses])

    pad = 1.5
    x_min, x_max = all_points[:, 0].min() - pad, all_points[:, 0].max() + pad
    y_min, y_max = all_points[:, 1].min() - pad, all_points[:, 1].max() + pad

    x = np.linspace(x_min, x_max, 120)
    y = np.linspace(y_min, y_max, 120)
    X, Y = np.meshgrid(x, y)

    U, _, _ = compute_multispring_fields(X, Y, springs)

    # Downsampled grid for clean force quiver arrows
    step = 10
    X_sub = X[::step, ::step]
    Y_sub = Y[::step, ::step]
    _, Fx_sub, Fy_sub = compute_multispring_fields(X_sub, Y_sub, springs)

    # ---------------------------------------------------------
    # Assemble Plotly Figure
    # ---------------------------------------------------------
    quiver_fig = ff.create_quiver(
        X_sub,
        Y_sub,
        Fx_sub,
        Fy_sub,
        scale=0.035 * scale,
        arrow_scale=0.25 * arrow_scale,
        line=dict(color="rgba(255, 255, 255, 0.4)", width=1.1),
        name="Net Vector Field",
    )

    for trace in quiver_fig.data:
        trace.showlegend = False

    fig = go.Figure(quiver_fig.data)

    # Potential Heatmap
    fig.add_trace(
        go.Heatmap(
            x=x,
            y=y,
            z=U,
            colorscale="Viridis",
            colorbar=dict(
                title=dict(text="Potential<br>U(x, y)", side="top"),
                x=1.16,
                len=0.9,
            ),
            opacity=0.85,
            showlegend=False,
            name="Potential Energy",
        )
    )

    # Potential Contours
    fig.add_trace(
        go.Contour(
            x=x,
            y=y,
            z=U,
            showscale=False,
            showlegend=False,
            contours=dict(coloring="none", showlines=True),
            line=dict(color="rgba(255, 255, 255, 0.2)", width=1),
            hoverinfo="skip",
        )
    )

    # Spring connection lines to EVERY mass
    spring_x, spring_y = [], []
    for m in masses:
        for s in springs:
            spring_x.extend([s["anchor"][0], m[0], None])
            spring_y.extend([s["anchor"][1], m[1], None])

    fig.add_trace(
        go.Scatter(
            x=spring_x,
            y=spring_y,
            mode="lines",
            line=dict(color="#facc15", width=1.8, dash="dot"),
            name="Springs",
        )
    )

    # Anchors
    anchor_names = [s.get("name", f"Anchor {i + 1}") for i, s in enumerate(springs)]
    fig.add_trace(
        go.Scatter(
            x=all_anchors[:, 0],
            y=all_anchors[:, 1],
            mode="markers+text",
            marker=dict(size=12, color="#ef4444", symbol="square"),
            text=anchor_names,
            textposition="top center",
            name="Anchors",
        )
    )

    # Masses
    mass_labels = [
        f"Mass {i + 1}" if len(masses) > 1 else "Mass" for i in range(len(masses))
    ]
    fig.add_trace(
        go.Scatter(
            x=masses[:, 0],
            y=masses[:, 1],
            mode="markers+text",
            marker=dict(size=14, color="#38bdf8", symbol="circle"),
            text=mass_labels,
            textposition="bottom right",
            name="Masses",
        )
    )

    # Compute net force and add an arrow for EACH mass
    for m in masses:
        _, Fx_pt, Fy_pt = compute_multispring_fields(
            np.array([m[0]]), np.array([m[1]]), springs
        )
        fx, fy = Fx_pt[0], Fy_pt[0]
        force_mag = np.hypot(fx, fy)

        # Only draw arrow if force is non-zero (avoids dot/degenerate artifacts)
        if force_mag > 1e-5:
            fig.add_annotation(
                x=m[0] + 0.07 * fx,
                y=m[1] + 0.07 * fy,
                ax=m[0],
                ay=m[1],
                xref="x",
                yref="y",
                axref="x",
                ayref="y",
                showarrow=True,
                arrowhead=force_arrow_head,
                arrowsize=force_arrow_size,
                arrowwidth=3,
                arrowcolor="#f97316",
            )

    # Legend proxy for the force arrows
    fig.add_trace(
        go.Scatter(
            x=[None],
            y=[None],
            mode="lines",
            line=dict(color="#f97316", width=3),
            name="Net Force",
        )
    )

    # ---------------------------------------------------------
    # Layout Configuration
    # ---------------------------------------------------------
    fig.update_layout(
        title=f"2D Potential & Force Field ({len(springs)} Springs, {len(masses)} Mass{'es' if len(masses) > 1 else ''})",
        xaxis=dict(title="x", range=[x_min, x_max], zeroline=False),
        yaxis=dict(
            title="y",
            range=[y_min, y_max],
            scaleanchor="x",
            scaleratio=1,
            zeroline=False,
        ),
        legend=dict(
            x=1.01,
            y=0.98,
            bgcolor="rgba(0,0,0,0.3)",
            bordercolor="rgba(255,255,255,0.2)",
            borderwidth=1,
        ),
        margin=dict(r=120),
        template="plotly_dark",
        width=980,
        height=650,
    )

    fig.show()

In [ ]:
springs = [
    {"anchor": np.array([-2.5, -1.0]), "k": 3.0, "L0": 1.2, "name": "Spring 1"},
    {"anchor": np.array([2.0, -1.5]), "k": 4.5, "L0": 1.0, "name": "Spring 2"},
    {"anchor": np.array([0.0, 2.5]), "k": 3.5, "L0": 1.5, "name": "Spring 3"},
    {"anchor": np.array([-1.5, 2.0]), "k": 2.0, "L0": 0.8, "name": "Spring 4"},
]

point_mass = np.array([0.2, 0.4])

plot_potential(springs, point_mass)

In [108]:
springs = [
    {"anchor": np.array([-2.5, -1.0]), "k": 3.0, "L0": 1.2, "name": "Spring 1"},
]

point_mass = np.array([0.2, 0.4])

plot_potential(springs, point_mass)

In [26]:
vec = np.array([1, 1.0])
point_mass = np.array([0.2, 0.4])

mag = np.sqrt(np.sum((vec - point_mass) ** 2))

springs = [
    {"anchor": vec, "k": 3.0, "L0": mag, "name": "Spring 1"},
]


plot_potential(springs, point_mass)

In [27]:
vec = np.array([1, 1.0])
point_mass = np.array([0.2, 0.4])

mag = np.sqrt(np.sum((vec - point_mass) ** 2)) - 1

springs = [
    {"anchor": vec, "k": 3.0, "L0": mag, "name": "Spring 1"},
]


plot_potential(springs, point_mass)

In [28]:
vec = np.array([1, 1.0])
point_mass = np.array([0.2, 0.4])

mag = np.sqrt(np.sum((vec - point_mass) ** 2)) + 1

springs = [
    {"anchor": vec, "k": 3.0, "L0": mag, "name": "Spring 1"},
]


plot_potential(springs, point_mass)

In [29]:
springs = [
    {"anchor": np.array([1, 1]), "k": 1, "L0": 1, "name": "Spring 1"},
    {"anchor": np.array([-1, -1]), "k": 1, "L0": 1, "name": "Spring 2"},
]

point_mass = np.array([0, 0])

plot_potential(springs, point_mass)

In [38]:
springs = [
    {"anchor": np.array([0, 1]), "k": 1, "L0": 1, "name": "Spring 1"},
    {"anchor": np.array([0, -1]), "k": 1, "L0": 1, "name": "Spring 2"},
]

point_mass = np.array([0, 1])

plot_potential(springs, point_mass)

In [100]:
point_masses = (np.array([1, 0]), np.array([0, 0]), np.array([.1, 0]))

spring1 = np.array([0, 1])

mag = np.sqrt(np.sum((spring1 - point_mass) ** 2))
print(mag)

springs = [
    {"anchor": np.array([0, 1]), "k": 3, "L0": 1, "name": "Spring 1"},
    {"anchor": np.array([0, -1]), "k": 3, "L0": 1, "name": "Spring 2"},
]


plot_potential(
    springs,
    point_masses,
    scale=0.1,
    arrow_scale=1.3,
    force_arrow_head=1,
    force_arrow_size=1,
)

1.4142135623730951


In [135]:
point_masses = (np.array([1, 0]))

spring1 = np.array([0, 1])

mag = np.sqrt(np.sum((spring1 - point_mass) ** 2))
print(mag)

springs = [
    {"anchor": np.array([0, 1]), "k": 3, "L0": 2, "name": "Spring 1"},
    {"anchor": np.array([0, -1]), "k": 3, "L0": 2, "name": "Spring 2"},
]


plot_potential(
    springs,
    point_masses,
    scale=1,
    arrow_scale=1.3,
    force_arrow_head=1,
    force_arrow_size=0.6,
)

0.6324555320336759


In [137]:
point_masses = (np.array([1, 0]), np.array([2.5, 0]))

spring1 = np.array([0, 1])

mag = np.sqrt(np.sum((spring1 - point_mass) ** 2))
print(mag)

springs = [
    {"anchor": np.array([0, 1]), "k": 3, "L0": 2, "name": "Spring 1"},
    {"anchor": np.array([0, -1]), "k": 3, "L0": 2, "name": "Spring 2"},
]


plot_potential(
    springs,
    point_masses,
    scale=1,
    arrow_scale=1.3,
    force_arrow_head=1,
    force_arrow_size=.6,
)

0.6324555320336759


In [138]:
point_masses = (
    np.array([1, 0]),
    np.array([2.5, 0]),
    np.array([-1, 0]),
    np.array([-2.5, 0]),
)

spring1 = np.array([0, 1])

mag = np.sqrt(np.sum((spring1 - point_mass) ** 2))
print(mag)

springs = [
    {"anchor": np.array([0, 1]), "k": 3, "L0": 2, "name": "Spring 1"},
    {"anchor": np.array([0, -1]), "k": 3, "L0": 2, "name": "Spring 2"},
]


plot_potential(
    springs,
    point_masses,
    scale=1,
    arrow_scale=1.3,
    force_arrow_head=1,
    force_arrow_size=0.6,
)

0.6324555320336759
